# Notebook 2: Parametric VaR — Normal, Student-t, EWMA
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '../src')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})

returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
port = returns @ weights
print(f'Loaded {len(port):,} observations | {returns.index[0].date()} to {returns.index[-1].date()}')


Loaded 2,609 observations | 2015-01-01 to 2024-12-31


## 1. Normal Distribution VaR

In [2]:
from var_models import normal_var
normal = normal_var(port)
print('Normal VaR:')
for k,v in normal.items(): print(f'  {k}: {v:.4f} ({v*100:.2f}%)')

Normal VaR:
  Normal_VaR_95pct: 0.0123 (1.23%)
  Normal_VaR_99pct: 0.0192 (1.92%)


## 2. Student-t VaR (Fat Tails)

In [3]:
from var_models import student_t_var
t_res = student_t_var(port)
print(f'Fitted degrees of freedom: {t_res["fitted_df"]:.2f}')
print()
for k,v in t_res.items():
    if k != 'fitted_df': print(f'  {k}: {v:.4f} ({v*100:.2f}%)')

Fitted degrees of freedom: 7.55

  StudentT_VaR_95pct: 0.0116 (1.16%)
  StudentT_VaR_99pct: 0.0211 (2.11%)


## 3. EWMA VaR

In [4]:
from var_models import ewma_var
ewma = ewma_var(port)
print(f'EWMA volatility: {ewma["ewma_vol"]:.6f}')
for k,v in ewma.items():
    if k != 'ewma_vol': print(f'  {k}: {v:.4f} ({v*100:.2f}%)')

EWMA volatility: 0.006783
  EWMA_VaR_95pct: 0.0113 (1.13%)
  EWMA_VaR_99pct: 0.0176 (1.76%)


## 4. Comparison Chart

In [5]:
labels=['95%','99%','99.5%']; cls=[0.95,0.99,0.995]
norm_v=[normal[f'Normal_VaR_{int(cl*100)}pct'] for cl in cls]
t_v=[t_res[f'StudentT_VaR_{int(cl*100)}pct'] for cl in cls]
ew_v=[ewma[f'EWMA_VaR_{int(cl*100)}pct'] for cl in cls]

import pandas as pd
comp=pd.DataFrame({'Normal VaR':norm_v,'Student-t VaR':t_v,'EWMA VaR':ew_v},index=labels)
comp.to_csv('../results/03_parametric_var_comparison.csv')
print(comp.round(4))

fig,ax=plt.subplots(figsize=(10,5))
x=np.arange(3); w=0.25
for i,(m,c,vals) in enumerate(zip(['Normal','Student-t','EWMA'],['#185FA5','#E24B4A','#1D9E75'],[norm_v,t_v,ew_v])):
    bars=ax.bar(x+(i-1)*w,vals,w,label=m,color=c,alpha=0.85)
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.0001,f'{v:.4f}',ha='center',va='bottom',fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(['95% VaR','99% VaR','99.5% VaR'])
ax.set_title('Parametric VaR: Normal vs Student-t vs EWMA'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/03_parametric_var_comparison.png',dpi=150,bbox_inches='tight')
plt.show()

       Normal VaR  Student-t VaR  EWMA VaR
95%        0.0123         0.0116    0.0113
99%        0.0192         0.0211    0.0176
99.5%      0.0192         0.0211    0.0176
